# Wearable Data Generator + Parquet vs CSV Benchmark POC

This notebook:
1. Generates realistic synthetic wearable/health streaming data
2. Saves it as both CSV and Parquet
3. Benchmarks query performance, file size, and memory usage

## 1. Configuration - Set Your Parameters Here

In [ ]:
# ============================================================
# USER CONFIGURATION
# ============================================================

NUM_CUSTOMERS   = 500        # number of unique customers/patients
NUM_ROWS        = 1_000_000  # total rows to generate
RANDOM_SEED     = 42

OUTPUT_CSV      = "wearable_data.csv"
OUTPUT_PARQUET  = "wearable_data.parquet"

# ============================================================
print(f"Target rows     : {NUM_ROWS:,}")
print(f"Customers       : {NUM_CUSTOMERS:,}")
print(f"Rows/customer   : {NUM_ROWS // NUM_CUSTOMERS:,} (approx)")

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import time
import os
import warnings
from faker import Faker
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")
np.random.seed(RANDOM_SEED)
fake = Faker()
Faker.seed(RANDOM_SEED)

print("Imports OK")

## 3. Scientific Reference Ranges

All ranges are based on published clinical and sports-science literature.

| Column | Normal Range | Source/Rationale |
|---|---|---|
| heart_rate_bpm | 40-180 bpm | Resting 60-100 (AHA); sleep 40-60; exercise up to 180 |
| spo2_pct | 92-100 % | Normal >= 95%; 92% is clinical lower threshold |
| steps | 0-250 steps/min | 0 at rest/sleep; brisk walk ~100; running ~160 |
| skin_temp_c | 30.0-37.5 C | Core ~37C; peripheral skin 30-35C (varies with activity) |
| hrv_ms | 15-120 ms | High HRV = rest/recovery; low HRV = stress/exercise |
| respiratory_rate | 8-30 breaths/min | Normal rest 12-20; sleep 8-14; heavy exercise up to 30 |

## 4. Activity Schedule by Hour

Activity probabilities are modelled on typical circadian rhythms.

In [ ]:
# Each hour (0-23) maps to a probability distribution over activities.
# Activities: sleeping, resting, walking, running, working, eating

ACTIVITIES = ["sleeping", "resting", "walking", "running", "working", "eating"]

# Rows = hour 0-23, Cols = activity probabilities (must sum to 1)
HOUR_ACTIVITY_PROBS = {
#   hour : [sleeping, resting, walking, running, working, eating]
    0  : [0.90, 0.08, 0.01, 0.00, 0.00, 0.01],
    1  : [0.92, 0.06, 0.01, 0.00, 0.00, 0.01],
    2  : [0.93, 0.05, 0.01, 0.00, 0.00, 0.01],
    3  : [0.93, 0.05, 0.01, 0.00, 0.00, 0.01],
    4  : [0.90, 0.07, 0.02, 0.00, 0.00, 0.01],
    5  : [0.75, 0.10, 0.08, 0.05, 0.01, 0.01],
    6  : [0.40, 0.15, 0.20, 0.10, 0.05, 0.10],
    7  : [0.15, 0.15, 0.25, 0.15, 0.10, 0.20],
    8  : [0.05, 0.10, 0.20, 0.10, 0.40, 0.15],
    9  : [0.02, 0.08, 0.20, 0.08, 0.55, 0.07],
    10 : [0.02, 0.08, 0.20, 0.08, 0.55, 0.07],
    11 : [0.02, 0.08, 0.20, 0.05, 0.50, 0.15],
    12 : [0.02, 0.10, 0.20, 0.03, 0.30, 0.35],
    13 : [0.05, 0.15, 0.20, 0.05, 0.40, 0.15],
    14 : [0.03, 0.10, 0.20, 0.05, 0.55, 0.07],
    15 : [0.02, 0.08, 0.20, 0.08, 0.55, 0.07],
    16 : [0.02, 0.08, 0.20, 0.10, 0.50, 0.10],
    17 : [0.02, 0.10, 0.25, 0.15, 0.30, 0.18],
    18 : [0.02, 0.10, 0.25, 0.10, 0.15, 0.38],
    19 : [0.03, 0.15, 0.25, 0.08, 0.10, 0.39],
    20 : [0.05, 0.35, 0.25, 0.05, 0.10, 0.20],
    21 : [0.10, 0.45, 0.20, 0.03, 0.07, 0.15],
    22 : [0.30, 0.45, 0.12, 0.01, 0.05, 0.07],
    23 : [0.60, 0.28, 0.06, 0.01, 0.02, 0.03],
}

print("Activity schedule defined for 24 hours")

## 5. Physiological Correlation Model

Each activity drives correlated values across all biometric columns.

In [ ]:
# Per-activity base stats: (mean, std) for each biometric
# Correlations are enforced by deriving values from shared noise terms

ACTIVITY_PROFILES = {
    #               hr_bpm        spo2_pct     steps        skin_temp_c   hrv_ms       resp_rate
    #               mean  std     mean  std    mean  std    mean  std     mean  std    mean  std
    "sleeping" : dict(hr=(55, 6),  spo2=(97.0, 0.8), steps=(0, 0),    temp=(33.5, 0.5), hrv=(95, 15),  rr=(12, 2)),
    "resting"  : dict(hr=(70, 8),  spo2=(98.0, 0.7), steps=(0, 2),    temp=(34.0, 0.5), hrv=(75, 12),  rr=(15, 2)),
    "eating"   : dict(hr=(75, 8),  spo2=(98.0, 0.6), steps=(5, 5),    temp=(34.2, 0.4), hrv=(65, 10),  rr=(16, 2)),
    "walking"  : dict(hr=(95, 12), spo2=(97.5, 0.8), steps=(100, 15), temp=(35.0, 0.6), hrv=(45, 10),  rr=(18, 3)),
    "working"  : dict(hr=(78, 10), spo2=(97.8, 0.7), steps=(10, 8),   temp=(34.0, 0.5), hrv=(60, 12),  rr=(16, 2)),
    "running"  : dict(hr=(155, 15),spo2=(96.0, 1.2), steps=(160, 20), temp=(36.5, 0.6), hrv=(22, 6),   rr=(26, 3)),
}

def generate_biometrics(activity, n):
    """
    Generate correlated biometric values for a given activity.
    A shared intensity noise term (0-1) drives all columns together
    so high heart rate coincides with high steps, low HRV, etc.
    """
    p = ACTIVITY_PROFILES[activity]

    # Shared intensity factor: same random draw influences all columns
    intensity = np.random.normal(0, 1, n)  # unit normal

    hr    = np.clip(p["hr"][0]   + p["hr"][1]   * intensity,          40,  200).round(0)
    spo2  = np.clip(p["spo2"][0] + p["spo2"][1] * (-intensity * 0.3), 92,  100).round(1)
    steps = np.clip(p["steps"][0]+ p["steps"][1]* np.abs(intensity),   0,  250).round(0).astype(int)
    temp  = np.clip(p["temp"][0] + p["temp"][1] * (intensity * 0.4),  30.0, 37.5).round(2)
    hrv   = np.clip(p["hrv"][0]  + p["hrv"][1]  * (-intensity * 0.6), 15,  120).round(1)
    rr    = np.clip(p["rr"][0]   + p["rr"][1]   * intensity,           8,   30).round(0).astype(int)

    return hr, spo2, steps, temp, hrv, rr

print("Physiological correlation model ready")

## 6. Generate Synthetic Data

In [ ]:
print(f"Generating {NUM_ROWS:,} rows for {NUM_CUSTOMERS} customers...")
t0 = time.time()

# --- Build customer registry ---
customer_ids   = [f"C{str(i).zfill(5)}" for i in range(1, NUM_CUSTOMERS + 1)]
patient_names  = [fake.name() for _ in range(NUM_CUSTOMERS)]
customer_map   = dict(zip(customer_ids, patient_names))

# --- Assign rows per customer (at least 1 per hour = 24 minimum) ---
rows_per_customer = max(NUM_ROWS // NUM_CUSTOMERS, 24)
actual_rows       = rows_per_customer * NUM_CUSTOMERS

print(f"Rows per customer : {rows_per_customer:,}")
print(f"Actual total rows : {actual_rows:,}")

# --- Generate per customer ---
all_chunks = []
base_start = datetime(2024, 1, 1, 0, 0, 0)

for cid in customer_ids:

    # One timestamp per hour, starting from a random offset within Jan 2024
    start_offset = np.random.randint(0, 30 * 24)   # random day offset up to 30 days
    start_ts     = base_start + timedelta(hours=int(start_offset))
    timestamps   = [start_ts + timedelta(hours=i) for i in range(rows_per_customer)]
    hours        = [ts.hour for ts in timestamps]

    # --- Assign activities based on hour probabilities ---
    activities = [
        np.random.choice(ACTIVITIES, p=HOUR_ACTIVITY_PROBS[h])
        for h in hours
    ]

    # --- Generate biometrics per activity group to exploit numpy vectorisation ---
    hr_arr   = np.zeros(rows_per_customer)
    spo2_arr = np.zeros(rows_per_customer)
    step_arr = np.zeros(rows_per_customer, dtype=int)
    temp_arr = np.zeros(rows_per_customer)
    hrv_arr  = np.zeros(rows_per_customer)
    rr_arr   = np.zeros(rows_per_customer, dtype=int)

    act_array = np.array(activities)
    for act in ACTIVITIES:
        mask = act_array == act
        n    = mask.sum()
        if n == 0:
            continue
        hr, spo2, steps, temp, hrv, rr = generate_biometrics(act, n)
        hr_arr[mask]   = hr
        spo2_arr[mask] = spo2
        step_arr[mask] = steps
        temp_arr[mask] = temp
        hrv_arr[mask]  = hrv
        rr_arr[mask]   = rr

    chunk = pd.DataFrame({
        "timestamp"        : timestamps,
        "customer_id"      : cid,
        "patient_name"     : customer_map[cid],
        "heart_rate_bpm"   : hr_arr.astype(int),
        "spo2_pct"         : spo2_arr,
        "steps"            : step_arr,
        "skin_temp_c"      : temp_arr,
        "hrv_ms"           : hrv_arr,
        "respiratory_rate" : rr_arr,
        "activity"         : activities,
        "source_file"      : f"stream_{cid}.json",
    })
    all_chunks.append(chunk)

df = pd.concat(all_chunks, ignore_index=True)

# Shuffle to simulate real streaming arrival order
df = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

elapsed = time.time() - t0
print(f"\nGenerated {len(df):,} rows in {elapsed:.2f}s")
df.head(5)

## 7. Data Quality Check

In [ ]:
print("=== Schema ===")
print(df.dtypes)
print()

print("=== Null counts ===")
print(df.isnull().sum())
print()

print("=== Biometric ranges ===")
cols = ["heart_rate_bpm", "spo2_pct", "steps", "skin_temp_c", "hrv_ms", "respiratory_rate"]
print(df[cols].agg(["min", "mean", "max"]).round(2))
print()

print("=== Activity distribution ===")
print(df["activity"].value_counts(normalize=True).round(3) * 100)

## 8. Save to CSV and Parquet

In [ ]:
print("Saving CSV...")
t0 = time.time()
df.to_csv(OUTPUT_CSV, index=False)
csv_write_time = time.time() - t0
csv_size_mb = os.path.getsize(OUTPUT_CSV) / 1024 / 1024
print(f"  CSV  : {csv_size_mb:.2f} MB  |  write time: {csv_write_time:.2f}s")

print("Saving Parquet (snappy)...")
t0 = time.time()
df.to_parquet(OUTPUT_PARQUET, index=False, compression="snappy")
pq_write_time = time.time() - t0
pq_size_mb = os.path.getsize(OUTPUT_PARQUET) / 1024 / 1024
print(f"  Parquet: {pq_size_mb:.2f} MB  |  write time: {pq_write_time:.2f}s")

print()
print(f"  Size reduction : {(1 - pq_size_mb / csv_size_mb) * 100:.1f}% smaller")
print(f"  Compression ratio: {csv_size_mb / pq_size_mb:.1f}x")

## 9. Benchmark - Query Performance

In [ ]:
REPEATS = 3  # run each query N times and take the average

def benchmark(fn, repeats=REPEATS):
    times = []
    for _ in range(repeats):
        t0 = time.time()
        result = fn()
        times.append(time.time() - t0)
    return result, sum(times) / len(times)

# ---- Query: avg heart rate during sleeping where result > 55 bpm ----

def query_csv():
    df_c = pd.read_csv(
        OUTPUT_CSV,
        usecols=["customer_id", "activity", "heart_rate_bpm"]
    )
    return (
        df_c[df_c["activity"] == "sleeping"]
        .groupby("customer_id")["heart_rate_bpm"]
        .mean()
        .pipe(lambda s: s[s > 55])
    )

def query_parquet():
    df_p = pd.read_parquet(
        OUTPUT_PARQUET,
        columns=["customer_id", "activity", "heart_rate_bpm"]
    )
    return (
        df_p[df_p["activity"] == "sleeping"]
        .groupby("customer_id")["heart_rate_bpm"]
        .mean()
        .pipe(lambda s: s[s > 55])
    )

print("Running CSV benchmark...")
res_csv, csv_query_time = benchmark(query_csv)

print("Running Parquet benchmark...")
res_pq, pq_query_time = benchmark(query_parquet)

print()
print(f"  CSV    query time (avg {REPEATS} runs): {csv_query_time:.4f}s")
print(f"  Parquet query time (avg {REPEATS} runs): {pq_query_time:.4f}s")
print(f"  Speedup: {csv_query_time / pq_query_time:.2f}x faster")

## 10. Benchmark - Memory Usage

In [ ]:
def mem_mb(df_):
    return df_.memory_usage(deep=True).sum() / 1024 / 1024

df_csv_full     = pd.read_csv(OUTPUT_CSV)
df_parquet_full = pd.read_parquet(OUTPUT_PARQUET)

csv_mem = mem_mb(df_csv_full)
pq_mem  = mem_mb(df_parquet_full)

print(f"  CSV    in-memory size : {csv_mem:.1f} MB")
print(f"  Parquet in-memory size: {pq_mem:.1f} MB")
print(f"  Parquet preserves typed dtypes: {df_parquet_full.dtypes.to_dict()}")

## 11. Full Summary Report

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "File size (MB)",
        "Write time (s)",
        "Query time - avg 3 runs (s)",
        "In-memory size (MB)",
        "Schema preserved",
        "Columnar storage",
    ],
    "CSV": [
        f"{csv_size_mb:.2f}",
        f"{csv_write_time:.2f}",
        f"{csv_query_time:.4f}",
        f"{csv_mem:.1f}",
        "No (all strings)",
        "No (row-based)",
    ],
    "Parquet": [
        f"{pq_size_mb:.2f}",
        f"{pq_write_time:.2f}",
        f"{pq_query_time:.4f}",
        f"{pq_mem:.1f}",
        "Yes (typed)",
        "Yes",
    ],
    "Winner": [
        "Parquet",
        "Parquet",
        "Parquet",
        "Similar",
        "Parquet",
        "Parquet",
    ]
})

print("=" * 65)
print("          CSV vs PARQUET - POC SUMMARY")
print("=" * 65)
print(summary.to_string(index=False))
print()
print(f"  Total rows      : {len(df):,}")
print(f"  Customers       : {NUM_CUSTOMERS}")
print(f"  Size reduction  : {(1 - pq_size_mb/csv_size_mb)*100:.1f}%")
print(f"  Query speedup   : {csv_query_time/pq_query_time:.2f}x")